In [ ]:
# --- Flow model: per-param-set 2-Gaussian-mixture fit (over log(Q), Q =
# network flow rate/discharge) + a small NN predicting the mixture's
# parameters (plus p0, the disconnected-network probability) from
# theta=(discspacing, length_ratio, b). Data: "3-parameter data" (10,000
# CSVs, one per param set, ~20,000 network trials each) -- this dataset's
# own parameter ranges (disc in ~[0.4,5], length_ratio in ~[0.03,0.5], b in
# ~[1e-5,3e-4]) already closely match the OG "transport" dataset used by
# the transport model in "[per p, t] ...]", even though the two datasets
# are independent random samples over that range (not the same param
# sets) -- so figures comparing the two models describe each one's own
# held-out set rather than claiming a literally shared one.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from scipy.stats import norm, wasserstein_distance
from pathlib import Path
import glob
import time

base_dir = Path("/Users/angelaliu/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/Fracture Flow UROP")
data_dir = base_dir / "3-parameter data"
outputs_dir = base_dir / "outputs"
outputs_dir.mkdir(exist_ok=True)


In [ ]:
def fit_gmm_logq_for_theta(theta_key, df_all, theta_cols, q_col="Q",
                           criterion="bic", random_state=0, n_init=10):
    """For one theta_key: empirical p0 = P(Q<=0), then fit a 2-component
    GMM to log(Q) for Q>0. Returns a dict with the true/fitted mixture
    parameters, the raw Q>0 samples (kept for the later NN-vs-truth
    Wasserstein metric), and logwd (Wasserstein distance between the GMM
    FIT's own samples and the true empirical samples -- a fit-quality
    diagnostic, distinct from the NN-vs-truth accuracy metric computed
    later)."""
    sub = df_all[df_all["theta_key"] == theta_key]
    q = sub[q_col].to_numpy(dtype=float)
    q = q[np.isfinite(q)]
    p0_emp = float(np.mean(q <= 0.0))
    q_pos = q[q > 0.0]
    theta = sub[theta_cols].iloc[0].to_numpy()

    if len(q_pos) < 2:
        return {"theta": theta, "p0_emp": p0_emp, "q_pos": q_pos, "K": 0,
                "w": np.array([]), "mu": np.array([]), "sigma": np.array([]), "logwd": np.nan}

    logq = np.log(q_pos).reshape(-1, 1)
    gmm = GaussianMixture(n_components=2, covariance_type="full",
                          random_state=random_state, n_init=n_init).fit(logq)

    w = gmm.weights_.copy()
    mu = gmm.means_.reshape(-1).copy()
    sigma = np.sqrt(np.array(gmm.covariances_).reshape(-1))
    order = np.argsort(mu)
    w, mu, sigma = w[order], mu[order], sigma[order]

    gmm_samples_logq, _ = gmm.sample(len(logq))
    gmm_samples_q_pos = np.exp(gmm_samples_logq).flatten()
    logwd = float(np.log(wasserstein_distance(gmm_samples_q_pos, q_pos)))

    return {"theta": theta, "p0_emp": p0_emp, "q_pos": q_pos, "K": 2,
            "w": w, "mu": mu, "sigma": sigma, "logwd": logwd}


In [ ]:
# --- Fit the GMM to a random subsample of param sets, not all 10,000 --
# fitting all 10,000 (each a ~20,000-row EM optimization) would take a
# very long time for no benefit to a summary figure. A large majority of
# this wide-parameter-range dataset's random draws land on fully- or
# mostly-disconnected networks (p0 near 1) -- verified directly: of a
# 20-file spot check, 14/20 had p0=1.0 exactly -- so FLOW_N_SAMPLE is set
# well above the final usable-set target to absorb that attrition, and the
# same P0_THRESHOLD=0.9 "usable" convention used throughout the transport
# model (`[per p, t] ...]`) is applied here too, not just the bare K==2
# fit-succeeded check.
FLOW_N_SAMPLE = 1000
P0_THRESHOLD = 0.9

all_csv_files = sorted(glob.glob(str(data_dir / "flow_outputs_*.csv")))
rng_files = np.random.default_rng(0)
sample_files = rng_files.choice(all_csv_files, size=min(FLOW_N_SAMPLE, len(all_csv_files)), replace=False)

all_file_indices = []
valid_file_indices = []
fit_results = {}  # file_idx -> fit dict from fit_gmm_logq_for_theta

t0 = time.time()
for fpath in sample_files:
    ind = int(Path(fpath).stem.split('_')[-1])
    df = pd.read_csv(fpath)
    df["theta_key"] = "k"
    fit = fit_gmm_logq_for_theta("k", df, ['discspacing', 'length_ratio', 'b'], q_col="Q")
    all_file_indices.append(ind)
    if fit["K"] == 2 and fit["p0_emp"] < P0_THRESHOLD:
        valid_file_indices.append(ind)
        fit_results[ind] = fit

print(f"[{time.time()-t0:.0f}s] fit {len(valid_file_indices)}/{len(sample_files)} sampled param sets successfully "
      f"(K=2 GMM, p0<{P0_THRESHOLD})")


In [ ]:
# --- Build the theta/output arrays (one row per successfully-fit param
# set, same order as valid_file_indices) and a held-out param-set split.
# Note this split is over THIS dataset's own param sets -- it cannot be
# made identical to the transport model's val_ps (different underlying
# dataset/sample), so figures describe each model's own held-out set.
theta_param_names = ["discspacing", "length_ratio", "b"]
output_param_names = ["p0", "w1", "mu1", "sigma1", "mu2", "sigma2", "logwd"]

thetas = np.array([fit_results[ind]["theta"] for ind in valid_file_indices], dtype=float)
outputs = np.array([
    [fit_results[ind]["p0_emp"], fit_results[ind]["w"][0],
     fit_results[ind]["mu"][0], fit_results[ind]["sigma"][0],
     fit_results[ind]["mu"][1], fit_results[ind]["sigma"][1],
     fit_results[ind]["logwd"]]
    for ind in valid_file_indices
], dtype=float)

thetas_means, thetas_stds = thetas.mean(axis=0), thetas.std(axis=0)
outputs_means, outputs_stds = outputs.mean(axis=0), outputs.std(axis=0)
thetas_norm = (thetas - thetas_means) / thetas_stds
outputs_norm = (outputs - outputs_means) / outputs_stds

rng_split = np.random.default_rng(42)
n_valid = len(valid_file_indices)
val_size = max(1, int(0.2 * n_valid))
val_pos = set(rng_split.choice(n_valid, size=val_size, replace=False).tolist())
is_val = np.array([i in val_pos for i in range(n_valid)])

thetas_train, thetas_val = thetas_norm[~is_val], thetas_norm[is_val]
outputs_train, outputs_val = outputs_norm[~is_val], outputs_norm[is_val]
print(f"flow model: {(~is_val).sum()} train param sets, {is_val.sum()} held-out param sets")


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

l2_reg = 9e-5
flow_model = keras.Sequential([
    layers.Input(shape=(3,)),
    layers.Dense(20, activation="relu", kernel_regularizer=regularizers.l2(l2_reg)),
    layers.Dense(20, activation="relu", kernel_regularizer=regularizers.l2(l2_reg)),
    layers.Dense(20, activation="relu", kernel_regularizer=regularizers.l2(l2_reg)),
    layers.Dense(20, activation="relu", kernel_regularizer=regularizers.l2(l2_reg)),
    layers.Dense(7),
])

# weights: [p0, w1, mu1, sigma1, mu2, sigma2, logwd] -- same weighting
# scheme as the original GMM.ipynb (mu/sigma weighted higher than p0/logwd)
loss_weights = tf.constant([1.0, 2.0, 3.0, 2.0, 3.0, 2.0, 0.5], dtype=tf.float32)
loss_weights = loss_weights / tf.reduce_sum(loss_weights)


def weighted_mse(y_true, y_pred):
    return tf.reduce_mean(tf.reduce_sum(loss_weights * tf.square(y_true - y_pred), axis=1))


def lr_by_epoch(epoch):
    if epoch < 500:
        return 1e-3
    elif epoch < 750:
        return 5e-4
    else:
        return 1e-4


flow_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss=weighted_mse)
t0 = time.time()
flow_history = flow_model.fit(
    thetas_train, outputs_train, validation_data=(thetas_val, outputs_val),
    batch_size=32, epochs=400,
    callbacks=[keras.callbacks.LearningRateScheduler(lr_by_epoch, verbose=0)], verbose=0,
)
print(f"[{time.time()-t0:.0f}s] flow_model -- final train loss: {flow_history.history['loss'][-1]:.4f}  "
      f"val loss: {flow_history.history['val_loss'][-1]:.4f}")
flow_model.save(outputs_dir / "flow_model.keras")


In [ ]:
# --- The accuracy metric the figures actually need: Wasserstein distance
# between the NN-PREDICTED mixture and the TRUE empirical log(Q) samples
# (drawing samples from the NN's predicted 2-Gaussian mixture, exponentiating
# back to Q-space, same convention as fit_gmm_logq_for_theta's own
# GMM-fit-vs-truth logwd) -- NOT the GMM-fit-vs-NN comparison the original
# GMM.ipynb computed (that compares two approximations to each other, not
# either one to the ground truth).
rng_wd = np.random.default_rng(1)


def nn_predict_mixture(theta_row):
    theta_norm = (np.array(theta_row) - thetas_means) / thetas_stds
    raw = flow_model(theta_norm.reshape(1, -1)).numpy().flatten() * outputs_stds + outputs_means
    p0_pred, w1_pred, mu1_pred, sigma1_pred, mu2_pred, sigma2_pred, logwd_pred = raw
    return p0_pred, w1_pred, mu1_pred, abs(sigma1_pred), mu2_pred, abs(sigma2_pred)


rows = []
for i, ind in enumerate(valid_file_indices):
    fit = fit_results[ind]
    disc, lr, b = fit["theta"]
    p0_true, w1_true, mu1_true, sigma1_true, mu2_true, sigma2_true = (
        fit["p0_emp"], fit["w"][0], fit["mu"][0], fit["sigma"][0], fit["mu"][1], fit["sigma"][1])
    p0_pred, w1_pred, mu1_pred, sigma1_pred, mu2_pred, sigma2_pred = nn_predict_mixture(fit["theta"])

    n = len(fit["q_pos"])
    comp = rng_wd.random(n) < w1_pred
    logq_samples = np.where(comp, rng_wd.normal(mu1_pred, sigma1_pred, n), rng_wd.normal(mu2_pred, sigma2_pred, n))
    nn_samples_q = np.exp(logq_samples)
    wd = float(wasserstein_distance(nn_samples_q, fit["q_pos"]))

    rows.append((ind, disc, lr, b, bool(is_val[i]),
                 p0_true, p0_pred, w1_true, w1_pred,
                 mu1_true, mu1_pred, sigma1_true, sigma1_pred,
                 mu2_true, mu2_pred, sigma2_true, sigma2_pred, wd))

flow_results_df = pd.DataFrame(rows, columns=[
    "paramset_idx", "discspacing", "length_ratio", "b", "is_val",
    "p0_true", "p0_pred", "w1_true", "w1_pred",
    "mu1_true", "mu1_pred", "sigma1_true", "sigma1_pred",
    "mu2_true", "mu2_pred", "sigma2_true", "sigma2_pred", "wd",
])
flow_results_df.to_csv(outputs_dir / "flow_results_df.csv", index=False)
print(f"Saved flow_results_df ({len(flow_results_df)} param sets, {flow_results_df['is_val'].sum()} held-out) "
      f"to {outputs_dir / 'flow_results_df.csv'}")
print(flow_results_df.loc[flow_results_df['is_val'], 'wd'].describe())
